- The current final model is model_partial3_4 — ResNet18 with Layer 3 + Layer 4 + FC trainable, saved as best_resnet18_partial3_4.pth. Its final reported result is 95.54% accuracy, 88.10% balanced accuracy, 91.39% Macro F1, and 0.9054 MCC.

- We will use the exact same seed, split, transforms, Dataset, and test loader from that notebook.

### Recreate the Exact Same Dataset

In [1]:
import os
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt

In [2]:
SEED = 42

IMAGE_SIZE = 224
NUM_CLASSES = 6
BATCH_SIZE = 32

CLASS_NAMES = [
    "MT_Blowhole",
    "MT_Break",
    "MT_Crack",
    "MT_Fray",
    "MT_Free",
    "MT_Uneven"
]

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [3]:
random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [4]:
PROJECT_ROOT = Path.cwd().parent

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "magnetic_tile"
)

print(DATASET_PATH)

image_paths = []
labels = []

for label, class_name in enumerate(CLASS_NAMES):

    image_folder = (
        DATASET_PATH
        / class_name
        / "Imgs"
    )

    paths = sorted(
        image_folder.glob("*.jpg")
    )

    image_paths.extend(paths)

    labels.extend(
        [label] * len(paths)
    )

print("Total images:", len(image_paths))

d:\Study\industrial_visual_inspection\data\raw\magnetic_tile
Total images: 1344


In [5]:
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    image_paths,
    labels,
    test_size=0.30,
    stratify=labels,
    random_state=SEED
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths,
    temp_labels,
    test_size=0.50,
    stratify=temp_labels,
    random_state=SEED
)

print("Train:", len(train_paths))
print("Validation:", len(val_paths))
print("Test:", len(test_paths))

test_distribution = Counter(test_labels)

print("Test class distribution:\n")

for class_idx, count in sorted(test_distribution.items()):
    print(
        f"{CLASS_NAMES[class_idx]:15s}: {count}"
    )

print("\nTest class percentages:\n")

total_test = len(test_labels)

for class_idx, count in sorted(test_distribution.items()):
    percentage = (
        count / total_test
    ) * 100

    print(
        f"{CLASS_NAMES[class_idx]:15s}: "
        f"{percentage:.2f}%"
    )

Train: 940
Validation: 202
Test: 202
Test class distribution:

MT_Blowhole    : 18
MT_Break       : 12
MT_Crack       : 9
MT_Fray        : 5
MT_Free        : 143
MT_Uneven      : 15

Test class percentages:

MT_Blowhole    : 8.91%
MT_Break       : 5.94%
MT_Crack       : 4.46%
MT_Fray        : 2.48%
MT_Free        : 70.79%
MT_Uneven      : 7.43%


In [6]:
# Recreating the Test Transformations
val_test_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.Grayscale(
        num_output_channels=3
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

In [7]:
# Dataset Class
class MagneticTileDataset(Dataset):

    def __init__(
        self,
        image_paths,
        labels,
        transform=None
    ):

        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):

        return len(self.image_paths)

    def __getitem__(self, idx):

        image = Image.open(
            self.image_paths[idx]
        ).convert("L")

        label = self.labels[idx]

        if self.transform is not None:

            image = self.transform(image)

        return image, label

test_dataset = MagneticTileDataset(
    test_paths,
    test_labels,
    val_test_transform
)

In [9]:
# Test DataLoader
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)
images, labels = next(
    iter(test_loader)
)

print("Images:", images.shape)
print("Labels:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)

Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32])
Image dtype: torch.float32
Label dtype: torch.int64


### Recreating the Best Model WE Found (ResNet-B0 With Partial Tuning Layer 3 and 4)

In [12]:
model_partial3_4 = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# Replace the final classification layer
model_partial3_4.fc = nn.Linear(
    model_partial3_4.fc.in_features,
    NUM_CLASSES
)

# Freeze the early layers
for param in model_partial3_4.conv1.parameters():
    param.requires_grad = False

for param in model_partial3_4.layer1.parameters():
    param.requires_grad = False

for param in model_partial3_4.layer2.parameters():
    param.requires_grad = False

# Keep deeper layers trainable
for param in model_partial3_4.layer3.parameters():
    param.requires_grad = True

for param in model_partial3_4.layer4.parameters():
    param.requires_grad = True

for param in model_partial3_4.fc.parameters():
    param.requires_grad = True

model_partial3_4 = model_partial3_4.to(device)

In [14]:
# Check Total Trainable Parameters
trainable_params = sum(
    p.numel()
    for p in model_partial3_4.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model_partial3_4.parameters()
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 11,179,590
Trainable parameters: 10,496,646


In [16]:
# Load the Best Checkpoints
checkpoint_path = (
    PROJECT_ROOT
    / "notebooks"
    / "best_resnet18_partial3_4.pth"
)

model_partial3_4.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device
    )
)

model_partial3_4.eval()

print("Best ResNet18 Partial FT model loaded.")

Best ResNet18 Partial FT model loaded.


In [17]:
# Run the Test Set Inference
all_true_labels = []
all_pred_labels = []
all_probabilities = []
all_image_paths = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model_partial3_4(images)

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        all_true_labels.extend(
            labels.cpu().numpy()
        )

        all_pred_labels.extend(
            predictions.cpu().numpy()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy()
        )

In [18]:
all_probabilities = np.array(
    all_probabilities
)

all_true_labels = np.array(
    all_true_labels
)

all_pred_labels = np.array(
    all_pred_labels
)

In [19]:
# Recheck the Basic Metrics
accuracy = accuracy_score(
    all_true_labels,
    all_pred_labels
)

balanced_accuracy = balanced_accuracy_score(
    all_true_labels,
    all_pred_labels
)

macro_f1 = f1_score(
    all_true_labels,
    all_pred_labels,
    average="macro"
)

mcc = matthews_corrcoef(
    all_true_labels,
    all_pred_labels
)

print(f"Accuracy:           {accuracy:.4f}")
print(f"Balanced Accuracy:  {balanced_accuracy:.4f}")
print(f"Macro F1:           {macro_f1:.4f}")
print(f"MCC:                {mcc:.4f}")

Accuracy:           0.9455
Balanced Accuracy:  0.8740
Macro F1:           0.8856
MCC:                0.8852


### Building the Error Analysis DataFrame 

In [22]:
predicted_confidence = (
    np.max(
        all_probabilities,
        axis=1
    )
)

top2_indices = np.argsort(
    all_probabilities,
    axis=1
)[:, -2:]

top2_indices = top2_indices[:, ::-1]

top1_indices = top2_indices[:, 0]
top2_indices_only = top2_indices[:, 1]

top1_probabilities = (
    all_probabilities[
        np.arange(len(all_probabilities)),
        top1_indices
    ]
)

top2_probabilities = (
    all_probabilities[
        np.arange(len(all_probabilities)),
        top2_indices_only
    ]
)

prediction_margin = (
    top1_probabilities -
    top2_probabilities
)
error_df = pd.DataFrame({

    "image_path": [
        str(path)
        for path in test_paths
    ],

    "true_label": all_true_labels,

    "true_class": [
        CLASS_NAMES[i]
        for i in all_true_labels
    ],

    "predicted_label": all_pred_labels,

    "predicted_class": [
        CLASS_NAMES[i]
        for i in all_pred_labels
    ],

    "confidence": predicted_confidence,

    "top2_class": [
        CLASS_NAMES[i]
        for i in top2_indices_only
    ],

    "top2_probability": top2_probabilities,

    "margin": prediction_margin,

    "correct": (
        all_true_labels ==
        all_pred_labels
    )
})

In [27]:
error_df.head()

,image_path,true_label,true_class,predicted_label,predicted_class,confidence,top2_class,top2_probability,margin,correct
0,d:\Study\industrial_visual_inspection\data\raw...,4,MT_Free,4,MT_Free,0.983771,MT_Blowhole,0.009838,0.973933,True
1,d:\Study\industrial_visual_inspection\data\raw...,5,MT_Uneven,5,MT_Uneven,0.999709,MT_Blowhole,0.000216,0.999493,True
2,d:\Study\industrial_visual_inspection\data\raw...,4,MT_Free,4,MT_Free,0.999415,MT_Break,0.000276,0.999138,True
3,d:\Study\industrial_visual_inspection\data\raw...,5,MT_Uneven,5,MT_Uneven,0.999815,MT_Break,0.000064,0.999751,True
4,d:\Study\industrial_visual_inspection\data\raw...,4,MT_Free,4,MT_Free,0.992314,MT_Crack,0.002944,0.989370,True


## Basic Error Summary

In [28]:
# Basic Error Summary
total_samples = len(error_df)

correct_samples = error_df["correct"].sum()

incorrect_samples = (
    total_samples -
    correct_samples
)

error_rate = (
    incorrect_samples /
    total_samples
)

print(
    f"Total samples:     {total_samples}"
)

print(
    f"Correct:            {correct_samples}"
)

print(
    f"Incorrect:          {incorrect_samples}"
)

print(
    f"Error rate:         {error_rate:.2%}"
)

Total samples:     202
Correct:            191
Incorrect:          11
Error rate:         5.45%


### We extract only error data

In [29]:
errors_df = (
    error_df[
        ~error_df["correct"]
    ]
    .copy()
    .sort_values(
        "confidence",
        ascending=False
    )
)

print(
    "Number of errors:",
    len(errors_df)
)

Number of errors: 11


## Que1. Which defect classes contribute most to the model's errors?

In [30]:
errors_by_true_class = (
    errors_df[
        "true_class"
    ]
    .value_counts()
    .reindex(
        CLASS_NAMES,
        fill_value=0
    )
)

errors_by_true_class

true_class
MT_Blowhole    0
MT_Break       5
MT_Crack       1
MT_Fray        1
MT_Free        4
MT_Uneven      0
Name: count, dtype: int64

- Raw numbers can be misleading as the samples of each defect class is different, so we calculate the error percentage

In [31]:
class_error_summary = (
    error_df
    .groupby("true_class")
    .agg(
        samples=("correct", "size"),
        errors=("correct", lambda x: (~x).sum())
    )
)

class_error_summary["error_rate"] = (
    class_error_summary["errors"]
    / class_error_summary["samples"]
)

class_error_summary = (
    class_error_summary
    .reindex(CLASS_NAMES)
)

class_error_summary

,samples,errors,error_rate
true_class,,,
MT_Blowhole,18,0,0.000000
MT_Break,12,5,0.416667
MT_Crack,9,1,0.111111
MT_Fray,5,1,0.200000
MT_Free,143,4,0.027972
MT_Uneven,15,0,0.000000


### Confusion Matrix

In [32]:
cm = confusion_matrix(
    all_true_labels,
    all_pred_labels,
    labels=range(NUM_CLASSES)
)

cm_df = pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
)

cm_df

,MT_Blowhole,MT_Break,MT_Crack,MT_Fray,MT_Free,MT_Uneven
MT_Blowhole,18,0,0,0,0,0
MT_Break,0,7,0,0,5,0
MT_Crack,0,1,8,0,0,0
MT_Fray,0,0,0,4,1,0
MT_Free,1,1,0,1,139,1
MT_Uneven,0,0,0,0,0,15


| Class        |     Recall | Main observation             |
| ------------ | ---------: | ---------------------------- |
| MT_Blowhole  |   **100%** | No test errors               |
| MT_Uneven    |   **100%** | No test errors               |
| MT_Free      | **97.20%** | 4 errors; absorbs Break/Fray |
| MT_Crack     | **88.89%** | 1 → Break                    |
| MT_Fray      | **80.00%** | 1 → Free                     |
| **MT_Break** | **58.33%** | **5 → Free**                 |


The model performs strongly across most defect classes, with the primary weakness occurring for MT_Break, where 5 of 12 test samples were classified as MT_Free. Other errors were sparse and distributed across MT_Crack, MT_Fray, and MT_Free. MT_Blowhole and MT_Uneven achieved zero classification errors on the test set.

In [34]:
print("ERROR ANALYSIS SUMMARY")
print("=" * 60)

print(
    f"Total test samples : {len(error_df)}"
)

print(
    f"Correct predictions: "
    f"{error_df['correct'].sum()}"
)

print(
    f"Misclassifications : "
    f"{(~error_df['correct']).sum()}"
)

print("\nMost affected classes:")

for class_name, row in (
    class_error_summary
    .sort_values(
        "error_rate",
        ascending=False
    )
    .iterrows()
):

    print(
        f"{class_name}: "
        f"{row['errors']:.0f} errors / "
        f"{row['samples']:.0f} samples "
        f"({row['error_rate']:.2%})"
    )

ERROR ANALYSIS SUMMARY
Total test samples : 202
Correct predictions: 191
Misclassifications : 11

Most affected classes:
MT_Break: 5 errors / 12 samples (41.67%)
MT_Fray: 1 errors / 5 samples (20.00%)
MT_Crack: 1 errors / 9 samples (11.11%)
MT_Free: 4 errors / 143 samples (2.80%)
MT_Blowhole: 0 errors / 18 samples (0.00%)
MT_Uneven: 0 errors / 15 samples (0.00%)
